In [16]:
import pandas as pd

pd.set_option('display.max_columns', 10)

In [17]:
import duckdb

con = duckdb.connect()

df = con.execute("""
SELECT
    e.id AS event_id,
    e.name AS event_name,
    COUNT(DISTINCT c.id) AS num_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    ON r.competition_id = c.id
JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
    ON e.id = r.event_id
WHERE
    c.country_id = 'Brazil'
    AND c.year = 2026
GROUP BY
    e.id,
    e.name
ORDER BY
    num_competitions DESC
""").df()

df

,event_id,event_name,num_competitions
0,333,3x3x3 Cube,32
1,222,2x2x2 Cube,30
2,clock,Clock,28
3,pyram,Pyraminx,26
4,minx,Megaminx,26
5,444,4x4x4 Cube,25
6,333oh,3x3x3 One-Handed,22
7,skewb,Skewb,21
8,333bf,3x3x3 Blindfolded,21
9,555,5x5x5 Cube,19


In [18]:
con = duckdb.connect()

df_camp_by_year = con.execute("""
SELECT
    year,
    COUNT(*) AS total_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t')
WHERE
    country_id = 'Brazil'
    AND cancelled = 0
GROUP BY year
ORDER BY year
""").df()


In [19]:
import duckdb

con = duckdb.connect()

df_final = pd.DataFrame()
for i in range(2006, 2027):

    df = con.execute(f"""
    SELECT
        e.id AS event_id,
        e.name AS event_name,
        COUNT(DISTINCT c.id) AS num_competitions
    FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
    JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
        ON r.competition_id = c.id
    JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
        ON e.id = r.event_id
    WHERE
        c.country_id = 'Brazil'
        AND c.year = {i}
    GROUP BY
        e.id,
        e.name
    ORDER BY
        num_competitions DESC
    """).df()

    df['year'] = i

    df_final = pd.concat([df_final, df])

df_final = df_final.merge(df_camp_by_year, on='year', how='left')

In [20]:
df_final.head()

,event_id,event_name,num_competitions,year,total_competitions
0,555,5x5x5 Cube,1,2007,1
1,333,3x3x3 Cube,1,2007,1
2,222,2x2x2 Cube,1,2007,1
3,minx,Megaminx,1,2007,1
4,444,4x4x4 Cube,1,2007,1


In [21]:
import plotly.graph_objects as go

def plot_events_over_time(df):

    fig = go.Figure()

    # linhas por modalidade
    for event_name in df["event_name"].unique():

        temp = (
            df[df["event_name"] == event_name]
            .sort_values("year")
        )

        fig.add_trace(
            go.Scatter(
                x=temp["year"],
                y=temp["num_competitions"],
                mode="lines+markers",
                name=event_name,
                hovertemplate=(
                    "<b>%{fullData.name}</b><br>"
                    "Ano: %{x}<br>"
                    "Competições: %{y}<extra></extra>"
                )
            )
        )

    # linha de total de competições
    total_df = (
        df[["year", "total_competitions"]]
        .drop_duplicates()
        .sort_values("year")
    )

    fig.add_trace(
        go.Scatter(
            x=total_df["year"],
            y=total_df["total_competitions"],
            mode="lines",
            name="Total de Competições",
            line=dict(width=5),
            hovertemplate=(
                "<b>Total de Competições</b><br>"
                "Ano: %{x}<br>"
                "Total: %{y}<extra></extra>"
            )
        )
    )

    fig.update_layout(
        title="Quantidade de Competições por Modalidade no Brasil",
        xaxis_title="Ano",
        yaxis_title="Número de Competições",
        hovermode="x unified",
        template="plotly_white",
        legend_title="Modalidade",
        height=700
    )

    return fig

In [22]:
def plot_event_presence_pct(df):

    fig = go.Figure()

    for event_name in df["event_name"].unique():

        temp = (
            df[df["event_name"] == event_name]
            .sort_values("year")
        )

        fig.add_trace(
            go.Scatter(
                x=temp["year"],
                y=temp["presence_pct"],
                mode="lines+markers",
                name=event_name
            )
        )

    fig.update_layout(
        title="Presença das Modalidades nas Competições Brasileiras",
        xaxis_title="Ano",
        yaxis_title="% das Competições",
        hovermode="x unified",
        template="plotly_white",
        height=700
    )

    return fig

In [30]:
df_final['presence_pct'] = df_final['num_competitions'] / df_final['total_competitions']

In [23]:
df_final = df_final[~df_final['event_name'].isin(['3x3x3 Multi-Blind Old Style', 'Magic', 'Master Magic'])]

In [24]:
df_final.sort_values(by='event_name', inplace=True)

In [27]:
fig_absolute = plot_events_over_time(df_final)
fig_absolute

In [31]:
fig_percentage = plot_event_presence_pct(df_final)
fig_percentage

In [32]:
from plotly.io import to_html

def export_dashboard(fig1, fig2, output_file="index.html"):

    graph1 = to_html(
        fig1,
        include_plotlyjs="cdn",
        full_html=False
    )

    graph2 = to_html(
        fig2,
        include_plotlyjs=False,
        full_html=False
    )

    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">

        <title>WCA Brasil - Análise de Modalidades</title>

        <style>
            body {{
                font-family: Arial, sans-serif;
                max-width: 1400px;
                margin: auto;
                padding: 20px;
            }}

            h1 {{
                text-align: center;
            }}

            .chart {{
                margin-top: 50px;
                margin-bottom: 50px;
            }}
        </style>
    </head>

    <body>

        <h1>
            Evolução das Modalidades da WCA no Brasil
        </h1>

        <p>
            Análise baseada nos dados públicos da WCA.
        </p>

        <div class="chart">
            <h2>
                Número de Competições por Modalidade
            </h2>
            {graph1}
        </div>

        <div class="chart">
            <h2>
                Percentual de Competições que Contêm Cada Modalidade
            </h2>
            {graph2}
        </div>

    </body>
    </html>
    """

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(html)

In [33]:
export_dashboard(
    fig_absolute,
    fig_percentage,
    "index.html"
)